**CELDA 1**

In [ ]:
!pip install findspark -q
!pip install --upgrade google-cloud-bigquery -q

import findspark
findspark.init()

from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, to_date, date_format,
    month, year, dayofmonth, quarter,
    when, trim, upper,
    count, sum as spark_sum, round as spark_round
)
from pyspark.sql.types import IntegerType, DecimalType

from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

spark = SparkSession.builder \
    .appName("ETL_TecnoMarket_ModeloEstrella") \
    .getOrCreate()

INPUT_PATH  = "/content/drive/MyDrive/Proyecto_Spark/"
OUTPUT_PATH = "/content/drive/MyDrive/Proyecto_Spark/modelo_estrella/"

Mounted at /content/drive


**CELDA 2**

In [ ]:
df_clientes_raw  = spark.read.option("header","true").option("inferSchema","true").csv(INPUT_PATH + "clientes.csv")
df_productos_raw = spark.read.option("header","true").option("inferSchema","true").csv(INPUT_PATH + "productos.csv")
df_geografia_raw = spark.read.option("header","true").option("inferSchema","true").csv(INPUT_PATH + "geografia.csv")
df_ventas_raw    = spark.read.option("header","true").option("inferSchema","true").csv(INPUT_PATH + "ventas.csv")

df_clientes_raw.show(truncate=False)
df_productos_raw.show(truncate=False)
df_geografia_raw.show(truncate=False)
df_ventas_raw.show(truncate=False)

+---------+-------------------+-----------------------+----------+
|IdCliente|Documento_Identidad|Nombre_Completo        |Segmento  |
+---------+-------------------+-----------------------+----------+
|1        |71234567           |Carlos Mendoza Ruiz    |VIP       |
|2        |45321678           |Ana Gabriel Flores     |Recurrente|
|3        |9876543            |Jean Pierre Claux      |Nuevo     |
|4        |76543210           |Milagros Soto Palomino |Recurrente|
|5        |41253698           |Renato Castillos Vega  |VIP       |
|6        |73418529           |Fiorella Beltrán Arce  |Nuevo     |
|7        |10293847           |Diego Torres Navarro   |Recurrente|
|8        |49586712           |Estefanía Wong Guerrero|VIP       |
|9        |70615243           |Pedro Picapiedra Prado |Nuevo     |
|10       |44332211           |Ximena Dávila Ugarte   |Recurrente|
|11       |55667788           |Luis Paredes Salas     |NULL      |
+---------+-------------------+-----------------------+-------

**CELDA 3**

In [ ]:
def reporte_nulos(df, nombre):
    total = df.count()
    print(f"\n{'='*50}")
    print(f"  {nombre}")
    print(f"{'='*50}")
    hay_nulos = False
    for c in df.columns:
        n = df.filter(col(c).isNull()).count()
        if n > 0:
            hay_nulos = True
            print(f"  ⚠  {c}: {n} nulo(s) ({round(n/total*100,1)}%)")
    if not hay_nulos:
        print("  ✓  Sin valores nulos.")
    print(f"  Total filas: {total}")

reporte_nulos(df_clientes_raw,  "CLIENTES  (crudo)")
reporte_nulos(df_productos_raw, "PRODUCTOS (crudo)")
reporte_nulos(df_geografia_raw, "GEOGRAFÍA (crudo)")
reporte_nulos(df_ventas_raw,    "VENTAS    (crudo)")


  CLIENTES  (crudo)
  ⚠  Segmento: 1 nulo(s) (9.1%)
  Total filas: 11

  PRODUCTOS (crudo)
  ⚠  Categoria: 1 nulo(s) (9.1%)
  Total filas: 11

  GEOGRAFÍA (crudo)
  ✓  Sin valores nulos.
  Total filas: 15

  VENTAS    (crudo)
  ✓  Sin valores nulos.
  Total filas: 35000


**CELDA 4**

In [ ]:
# ── dim_clientes ───────────────────────────────────────────────────
dim_clientes = (
    df_clientes_raw
    .withColumn("Documento_Identidad", trim(col("Documento_Identidad")))
    .withColumn("Nombre_Completo", trim(col("Nombre_Completo")))
    .withColumn(
        "Segmento",
        when(
            col("Segmento").isin("Nuevo", "Recurrente", "VIP"),
            col("Segmento")
        ).otherwise(lit("Nuevo"))
    )
    .filter(col("Documento_Identidad").isNotNull())
    .select(
        col("IdCliente").cast(IntegerType()),
        col("Documento_Identidad"),
        col("Nombre_Completo"),
        col("Segmento")
    )
    .distinct()
)

# ── dim_productos ──────────────────────────────────────────────────
dim_productos = (
    df_productos_raw
    .withColumn("SKU", upper(trim(col("SKU"))))
    .withColumn("NombreProducto", trim(col("NombreProducto")))
    .withColumn(
        "Categoria",
        when(
            col("Categoria").isNull() |
            (trim(col("Categoria")) == ""),
            lit("Sin Categoría")
        ).otherwise(trim(col("Categoria")))
    )
    .withColumn("Marca", trim(col("Marca")))
    .filter(col("SKU").isNotNull())
    .select(
        col("IdProducto").cast(IntegerType()),
        col("SKU"),
        col("NombreProducto"),
        col("Categoria"),
        col("Marca")
    )
    .distinct()
)

# ── dim_geografia ──────────────────────────────────────────────────
dim_geografia = (
    df_geografia_raw
    .withColumn("Departamento", trim(col("Departamento")))
    .withColumn("Provincia", trim(col("Provincia")))
    .withColumn("Distrito", trim(col("Distrito")))
    .withColumn(
        "Zona_Logistica",
        when(
            col("Zona_Logistica").isin("Lima", "Norte", "Sur", "Centro"),
            col("Zona_Logistica")
        ).otherwise(lit("Centro"))
    )
    .filter(
        col("Departamento").isNotNull() &
        col("Provincia").isNotNull() &
        col("Distrito").isNotNull()
    )
    .select(
        col("IdGeografia").cast(IntegerType()),
        col("Departamento"),
        col("Provincia"),
        col("Distrito"),
        col("Zona_Logistica")
    )
    .distinct()
)

reporte_nulos(dim_clientes, "dim_clientes  (limpia)")
reporte_nulos(dim_productos, "dim_productos (limpia)")
reporte_nulos(dim_geografia, "dim_geografia (limpia)")

dim_clientes.show(truncate=False)
dim_productos.show(truncate=False)
dim_geografia.show(truncate=False)


  dim_clientes  (limpia)
  ✓  Sin valores nulos.
  Total filas: 11

  dim_productos (limpia)
  ✓  Sin valores nulos.
  Total filas: 11

  dim_geografia (limpia)
  ✓  Sin valores nulos.
  Total filas: 15
+---------+-------------------+-----------------------+----------+
|IdCliente|Documento_Identidad|Nombre_Completo        |Segmento  |
+---------+-------------------+-----------------------+----------+
|2        |45321678           |Ana Gabriel Flores     |Recurrente|
|11       |55667788           |Luis Paredes Salas     |Nuevo     |
|8        |49586712           |Estefanía Wong Guerrero|VIP       |
|9        |70615243           |Pedro Picapiedra Prado |Nuevo     |
|7        |10293847           |Diego Torres Navarro   |Recurrente|
|1        |71234567           |Carlos Mendoza Ruiz    |VIP       |
|4        |76543210           |Milagros Soto Palomino |Recurrente|
|3        |9876543            |Jean Pierre Claux      |Nuevo     |
|6        |73418529           |Fiorella Beltrán Arce  |Nuev

**CELDA 5**

In [ ]:
# ── Dimensiones estáticas ──────────────────────────────────────────
dim_sucursales = spark.createDataFrame([
    Row(IdSucursal=1, Nombre_Sucursal="Plataforma Web Nacional", Canal="E-commerce"),
    Row(IdSucursal=2, Nombre_Sucursal="Tienda Física CC Jockey Plaza", Canal="Presencial"),
    Row(IdSucursal=3, Nombre_Sucursal="Tienda Física Real Plaza Arequipa", Canal="Presencial"),
    Row(IdSucursal=4, Nombre_Sucursal="Tienda Física Mall Aventura Trujillo", Canal="Presencial"),
])

dim_promociones = spark.createDataFrame([
    Row(IdPromocion=1, Nombre_Promo="Sin Promoción / Precio Regular", Tipo_Descuento="Monto Fijo", Valor_Descuento=0.00),
    Row(IdPromocion=2, Nombre_Promo="Campaña Cyber Wow Mayo", Tipo_Descuento="Porcentaje", Valor_Descuento=15.00),
    Row(IdPromocion=3, Nombre_Promo="Descuento Black Friday", Tipo_Descuento="Porcentaje", Valor_Descuento=20.00),
    Row(IdPromocion=4, Nombre_Promo="Cupón Gamer Extremo", Tipo_Descuento="Monto Fijo", Valor_Descuento=100.00),
    Row(IdPromocion=5, Nombre_Promo="Campaña Back to School", Tipo_Descuento="Porcentaje", Valor_Descuento=10.00),
])

dim_metodos_pago = spark.createDataFrame([
    Row(IdMetodoPago=1, Descripcion="Tarjeta Credito"),
    Row(IdMetodoPago=2, Descripcion="Tarjeta Debito"),
    Row(IdMetodoPago=3, Descripcion="Yape"),
    Row(IdMetodoPago=4, Descripcion="Plin"),
    Row(IdMetodoPago=5, Descripcion="Efectivo"),
])

# ── dim_tiempo ─────────────────────────────────────────────────────
dim_tiempo = (
    df_ventas_raw
    .withColumn(
        "Fecha",
        when(
            to_date(col("FechaVenta"), "yyyy-MM-dd").isNotNull(),
            to_date(col("FechaVenta"), "yyyy-MM-dd")
        )
        .when(
            to_date(col("FechaVenta"), "dd/MM/yyyy").isNotNull(),
            to_date(col("FechaVenta"), "dd/MM/yyyy")
        )
        .when(
            to_date(col("FechaVenta"), "MM/dd/yyyy").isNotNull(),
            to_date(col("FechaVenta"), "MM/dd/yyyy")
        )
        .when(
            to_date(col("FechaVenta"), "dd-MM-yyyy").isNotNull(),
            to_date(col("FechaVenta"), "dd-MM-yyyy")
        )
        .otherwise(None)
    )
    .filter(col("Fecha").isNotNull())
    .select("Fecha", "Es_Evento")
    .distinct()
    .withColumn(
        "IdTiempo",
        date_format(col("Fecha"), "yyyyMMdd").cast(IntegerType())
    )
    .withColumn("Anio", year(col("Fecha")))
    .withColumn("Mes", month(col("Fecha")))
    .withColumn("Dia", dayofmonth(col("Fecha")))
    .withColumn("Trimestre", quarter(col("Fecha")))
    .withColumn("Es_Evento", col("Es_Evento").cast(IntegerType()))
    .select(
        "IdTiempo",
        "Fecha",
        "Anio",
        "Mes",
        "Dia",
        "Trimestre",
        "Es_Evento"
    )
    .orderBy("Fecha")
)

dim_tiempo.show(truncate=False)

+--------+----------+----+---+---+---------+---------+
|IdTiempo|Fecha     |Anio|Mes|Dia|Trimestre|Es_Evento|
+--------+----------+----+---+---+---------+---------+
|20230101|2023-01-01|2023|1  |1  |1        |1        |
|20230101|2023-01-01|2023|1  |1  |1        |0        |
|20230102|2023-01-02|2023|1  |2  |1        |0        |
|20230102|2023-01-02|2023|1  |2  |1        |1        |
|20230103|2023-01-03|2023|1  |3  |1        |1        |
|20230103|2023-01-03|2023|1  |3  |1        |0        |
|20230104|2023-01-04|2023|1  |4  |1        |0        |
|20230104|2023-01-04|2023|1  |4  |1        |1        |
|20230105|2023-01-05|2023|1  |5  |1        |0        |
|20230105|2023-01-05|2023|1  |5  |1        |1        |
|20230106|2023-01-06|2023|1  |6  |1        |0        |
|20230106|2023-01-06|2023|1  |6  |1        |1        |
|20230107|2023-01-07|2023|1  |7  |1        |0        |
|20230107|2023-01-07|2023|1  |7  |1        |1        |
|20230108|2023-01-08|2023|1  |8  |1        |1        |
|20230108|

**CELDA 6**

In [ ]:
df_ventas_clean = df_ventas_raw \
    .withColumn("FechaVenta",      to_date(col("FechaVenta"), "yyyy-MM-dd")) \
    .withColumn("Cantidad",        col("Cantidad").cast(IntegerType())) \
    .withColumn("Monto_Bruto",     col("Monto_Bruto").cast(DecimalType(12,2))) \
    .withColumn("Monto_Descuento",
        when(col("Monto_Descuento").isNull(), lit(0.0))
        .otherwise(col("Monto_Descuento").cast(DecimalType(12,2)))
    ) \
    .withColumn("Monto_Neto", col("Monto_Bruto") - col("Monto_Descuento")) \
    .withColumn("IdTiempo", date_format(col("FechaVenta"), "yyyyMMdd").cast(IntegerType())) \
    .filter(
        col("FechaVenta").isNotNull()          &
        col("SKU").isNotNull()                 &
        col("Documento_Identidad").isNotNull() &
        col("Monto_Bruto").isNotNull()         &
        (col("Cantidad") >= 0)
    )

fact_ventas = df_ventas_clean \
    .join(dim_productos.select("SKU","IdProducto"),
          on="SKU", how="left") \
    .join(dim_clientes.select("Documento_Identidad","IdCliente"),
          on="Documento_Identidad", how="left") \
    .join(dim_geografia.select("Departamento","Provincia","Distrito","IdGeografia"),
          on=["Departamento","Provincia","Distrito"], how="left") \
    .join(dim_sucursales.select("Nombre_Sucursal","IdSucursal"),
          on="Nombre_Sucursal", how="left") \
    .join(dim_promociones.select("Nombre_Promo","IdPromocion"),
          on="Nombre_Promo", how="left") \
    .join(dim_metodos_pago.select("Descripcion","IdMetodoPago"),
          on="Descripcion", how="left") \
    .select(
        col("IdVenta").cast(IntegerType()),
        col("IdProducto"), col("IdCliente"),  col("IdTiempo"),
        col("IdGeografia"),col("IdSucursal"), col("IdPromocion"), col("IdMetodoPago"),
        col("Cantidad"), col("Monto_Bruto"), col("Monto_Descuento"), col("Monto_Neto")
    ) \
    .filter(
        col("IdProducto").isNotNull()  &
        col("IdCliente").isNotNull()   &
        col("IdGeografia").isNotNull() &
        col("IdSucursal").isNotNull()  &
        col("IdPromocion").isNotNull() &
        col("IdMetodoPago").isNotNull()
    )

reporte_nulos(fact_ventas, "fact_ventas (final)")
fact_ventas.show(truncate=False)


  fact_ventas (final)
  ✓  Sin valores nulos.
  Total filas: 1838
+-------+----------+---------+--------+-----------+----------+-----------+------------+--------+-----------+---------------+----------+
|IdVenta|IdProducto|IdCliente|IdTiempo|IdGeografia|IdSucursal|IdPromocion|IdMetodoPago|Cantidad|Monto_Bruto|Monto_Descuento|Monto_Neto|
+-------+----------+---------+--------+-----------+----------+-----------+------------+--------+-----------+---------------+----------+
|60     |10        |6        |20250416|11         |1         |3          |5           |5       |1950.00    |390.0          |1560.0    |
|86     |3         |5        |20250323|2          |1         |3          |5           |7       |35700.00   |7140.0         |28560.0   |
|134    |10        |7        |20241010|11         |1         |3          |5           |7       |2730.00    |546.0          |2184.0    |
|176    |6         |9        |20250528|6          |1         |3          |5           |4       |840.00     |168.0    

**CELDA 7**

In [ ]:
# ── Crecimiento YoY ────────────────────────────────────────────────
(
    fact_ventas
    .join(
        dim_tiempo.select("IdTiempo", "Anio"),
        on="IdTiempo"
    )
    .join(
        dim_productos.select("IdProducto", "Categoria"),
        on="IdProducto"
    )
    .groupBy("Anio", "Categoria")
    .agg(
        spark_sum("Cantidad").alias("Unidades_Vendidas"),
        spark_round(spark_sum("Monto_Bruto"), 2).alias("Facturacion_Bruta"),
        spark_round(spark_sum("Monto_Descuento"), 2).alias("Descuentos_Absorbidos"),
        spark_round(spark_sum("Monto_Neto"), 2).alias("Ingreso_Neto_Real")
    )
    .orderBy("Anio", "Categoria")
    .show(truncate=False)
)

# ── Inteligencia Territorial ───────────────────────────────────────
(
    fact_ventas
    .join(
        dim_geografia.select(
            "IdGeografia",
            "Zona_Logistica",
            "Departamento"
        ),
        on="IdGeografia"
    )
    .groupBy(
        "Zona_Logistica",
        "Departamento"
    )
    .agg(
        count("IdVenta").alias("Total_Transacciones"),
        spark_sum("Cantidad").alias("Productos_Despachados"),
        spark_round(
            spark_sum("Monto_Neto"),
            2
        ).alias("Venta_Neta_Total")
    )
    .withColumn(
        "Ticket_Promedio_Soles",
        spark_round(
            col("Venta_Neta_Total") /
            col("Total_Transacciones"),
            2
        )
    )
    .orderBy(
        "Zona_Logistica",
        "Departamento"
    )
    .show(truncate=False)
)

+----+--------------+-----------------+-----------------+---------------------+-----------------+
|Anio|Categoria     |Unidades_Vendidas|Facturacion_Bruta|Descuentos_Absorbidos|Ingreso_Neto_Real|
+----+--------------+-----------------+-----------------+---------------------+-----------------+
|2023|Almacenamiento|1254             |597120.00        |119424.0             |477696.0         |
|2023|Componentes   |1164             |923400.00        |184680.0             |738720.0         |
|2023|Laptops       |1976             |7855200.00       |1571040.0            |6284160.0        |
|2023|Monitores     |626              |594700.00        |118940.0             |475760.0         |
|2023|Periféricos   |1258             |429020.00        |85804.0              |343216.0         |
|2023|Sin Categoría |850              |807500.00        |161500.0             |646000.0         |
|2024|Almacenamiento|1218             |588480.00        |117696.0             |470784.0         |
|2024|Componentes   

**CELDA 8**

In [ ]:
tablas = {
    "dim_clientes": dim_clientes,
    "dim_productos": dim_productos,
    "dim_geografia": dim_geografia,
    "dim_tiempo": dim_tiempo,
    "dim_sucursales": dim_sucursales,
    "dim_promociones": dim_promociones,
    "dim_metodos_pago": dim_metodos_pago,
    "fact_ventas": fact_ventas,
}

for nombre, df in tablas.items():
    (
        df.coalesce(1)
        .write
        .option("header", "true")
        .mode("overwrite")
        .csv(OUTPUT_PATH + nombre)
    )

spark.stop()

**CELDA 9**

In [ ]:
import glob
from google.cloud import bigquery

PROJECT_ID = "proyecto-etl-tm"
DATASET_ID = "tecnomarket_etl"
client = bigquery.Client(project=PROJECT_ID)

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    create_disposition=bigquery.CreateDisposition.CREATE_IF_NEEDED,
)

for nombre in tablas:
    patron = OUTPUT_PATH + nombre + "/part-*.csv"
    archivos = glob.glob(patron)
    ruta_csv = archivos[0]

    with open(ruta_csv, "rb") as f:
        job = client.load_table_from_file(
            f,
            f"{PROJECT_ID}.{DATASET_ID}.{nombre}",
            job_config=job_config
        )
        job.result()